In [13]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. 데이터 로드
# ============================================================
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
submission = pd.read_csv('/content/sample_submission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Target 분포: mean={train['stress_score'].mean():.3f}, std={train['stress_score'].std():.3f}")

target = train['stress_score'].copy()
train_id = train['ID'].copy()
test_id = test['ID'].copy()

# train/test 합쳐서 전처리
train.drop(['ID', 'stress_score'], axis=1, inplace=True)
test.drop(['ID'], axis=1, inplace=True)
data = pd.concat([train, test], axis=0, ignore_index=True)

print(f"\nCombined data: {data.shape}")

# ============================================================
# 2. 피처 엔지니어링
# ============================================================
print("\n[피처 엔지니어링]")

# 2-1. BMI (체질량지수) - 건강 지표로 스트레스와 상관관계
data['bmi'] = data['weight'] / ((data['height']/100) ** 2)

# 2-2. 혈압 관련 파생변수
data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']  # 맥압
data['mean_arterial_pressure'] = data['diastolic_blood_pressure'] + data['pulse_pressure'] / 3  # 평균동맥압
data['bp_ratio'] = data['systolic_blood_pressure'] / (data['diastolic_blood_pressure'] + 1e-5)  # 수축기/이완기 비율

# 2-3. 고혈압 여부 (수축기>=140 또는 이완기>=90)
data['hypertension'] = ((data['systolic_blood_pressure'] >= 140) |
                         (data['diastolic_blood_pressure'] >= 90)).astype(int)

# 2-4. 콜레스테롤 수준 (정상/경계/높음)
data['cholesterol_level'] = pd.cut(data['cholesterol'], bins=[0, 200, 240, 500], labels=[0, 1, 2]).astype(int)

# 2-5. 혈당 수준 (정상/전당뇨/당뇨)
data['glucose_level'] = pd.cut(data['glucose'], bins=[0, 100, 126, 300], labels=[0, 1, 2]).astype(int)

# 2-6. 골밀도 수준
data['bone_risk'] = (data['bone_density'] < 0.5).astype(int)

# 2-7. 나이 그룹
data['age_group'] = pd.cut(data['age'], bins=[0, 30, 45, 60, 75, 100], labels=[0, 1, 2, 3, 4]).astype(int)

# 2-8. 비만도 (BMI 기반)
data['obesity'] = pd.cut(data['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)

# 2-9. 건강 리스크 종합 점수
data['health_risk_score'] = (data['hypertension'] + data['cholesterol_level'] +
                              data['glucose_level'] + data['bone_risk'] + data['obesity'])

# 2-10. 나이*활동량 교호작용 (나이 많고 활동 적으면 스트레스 높을 수 있음)
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
data['activity_num'] = data['activity'].map(activity_map)
data['age_activity'] = data['age'] * data['activity_num']

# 2-11. 수치형 변수 간 상호작용
data['age_bmi'] = data['age'] * data['bmi']
data['cholesterol_glucose'] = data['cholesterol'] * data['glucose']

print(f"  파생변수 생성 완료. 현재 컬럼 수: {data.shape[1]}")

# ============================================================
# 3. 결측치 처리
# ============================================================
print("\n[결측치 처리]")

# medical_history: NaN → 'none' (없는 것도 정보)
data['medical_history'] = data['medical_history'].fillna('none')
data['family_medical_history'] = data['family_medical_history'].fillna('none')

# edu_level: NaN → 'unknown'
data['edu_level'] = data['edu_level'].fillna('unknown')

# mean_working: 34% 결측 → 중앙값 대체 + 결측 여부 피처
data['mean_working_missing'] = data['mean_working'].isnull().astype(int)
data['mean_working'] = data['mean_working'].fillna(data['mean_working'].median())

print(f"  결측치 처리 완료. 남은 결측: {data.isnull().sum().sum()}")

# ============================================================
# 4. 카테고리 인코딩
# ============================================================
print("\n[카테고리 인코딩]")

# Ordinal Encoding (SVR에 적합하도록)
cat_cols = ['gender', 'activity', 'smoke_status', 'medical_history',
            'family_medical_history', 'sleep_pattern', 'edu_level']

# Label Encoding
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    le_dict[col] = le

print(f"  인코딩 완료. 최종 컬럼: {list(data.columns)}")

# ============================================================
# 5. Train/Test 분리
# ============================================================
X_train = data.iloc[:len(train)].copy()
X_test = data.iloc[len(train):].copy()
y_train = target.copy()

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")

# ============================================================
# 6. 스케일링 (SVR에 필수)
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_names = X_train.columns.tolist()

# ============================================================
# 7. 모델 정의 + CV 평가 함수
# ============================================================
def evaluate_model(model, X, y, model_name="Model", n_splits=5):
    """K-Fold CV로 MAE 평가"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    mae_scores = []
    oof_preds = np.zeros(len(y))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y.values[train_idx], y.values[val_idx]

        model_clone = __import__('copy').deepcopy(model)
        model_clone.fit(X_tr, y_tr)
        preds = model_clone.predict(X_val)

        # 클리핑 (0~1 범위)
        preds = np.clip(preds, 0, 1)
        oof_preds[val_idx] = preds

        mae = mean_absolute_error(y_val, preds)
        mae_scores.append(mae)

    mean_mae = np.mean(mae_scores)
    std_mae = np.std(mae_scores)
    print(f"  {model_name}: MAE = {mean_mae:.5f} (±{std_mae:.5f})")
    return mean_mae, oof_preds

# ============================================================
# 8. 개별 모델 학습 + 평가
# ============================================================
print("\n" + "="*60)
print("[개별 모델 CV 평가]")
print("="*60)

# --- (1) SVR  ---
# SVR은 RBF 커널이 기본, C/epsilon/gamma 튜닝이 핵심
svr_model = SVR(kernel='rbf', C=10, epsilon=0.01, gamma='scale')
svr_mae, svr_oof = evaluate_model(svr_model, X_train_scaled, y_train, "SVR (RBF)")

# SVR with poly kernel
svr_poly = SVR(kernel='poly', C=5, epsilon=0.01, degree=3, gamma='scale')
svr_poly_mae, svr_poly_oof = evaluate_model(svr_poly, X_train_scaled, y_train, "SVR (Poly)")

# --- (2) Ridge Regression ---
ridge_model = Ridge(alpha=1.0)
ridge_mae, ridge_oof = evaluate_model(ridge_model, X_train_scaled, y_train, "Ridge")

# --- (3) Gradient Boosting ---
gbr_model = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=42
)
gbr_mae, gbr_oof = evaluate_model(gbr_model, X_train_scaled, y_train, "GBR")

# --- (4) CatBoost  ---
try:
    from catboost import CatBoostRegressor
    cat_model = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=42, verbose=0,
        loss_function='MAE'
    )
    cat_mae, cat_oof = evaluate_model(cat_model, X_train_scaled, y_train, "CatBoost")
    HAS_CATBOOST = True
except ImportError:
    print("  CatBoost not available, skipping...")
    HAS_CATBOOST = False

# --- (5) XGBoost ---
try:
    from xgboost import XGBRegressor
    xgb_model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=42, verbosity=0
    )
    xgb_mae, xgb_oof = evaluate_model(xgb_model, X_train_scaled, y_train, "XGBoost")
    HAS_XGB = True
except ImportError:
    print("  XGBoost not available, skipping...")
    HAS_XGB = False

# --- (6) LightGBM ---
try:
    from lightgbm import LGBMRegressor
    lgbm_model = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1
    )
    lgbm_mae, lgbm_oof = evaluate_model(lgbm_model, X_train_scaled, y_train, "LightGBM")
    HAS_LGBM = True
except ImportError:
    print("  LightGBM not available, skipping...")
    HAS_LGBM = False

# ============================================================
# 9. Seed Ensemble  + 모델 앙상블
# ============================================================
print("\n" + "="*60)
print("[Seed Ensemble + 모델 앙상블 예측]")
print("="*60)

def seed_ensemble_predict(model_class, model_params, X_train, y_train, X_test, seeds, model_name="Model"):
    """여러 시드로 학습 후 평균 예측 (안정성 확보)"""
    all_preds = []
    for seed in seeds:
        params = model_params.copy()
        if 'random_state' in params or 'random_seed' in params:
            if 'random_seed' in params:
                params['random_seed'] = seed
            else:
                params['random_state'] = seed

        model = model_class(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        preds = np.clip(preds, 0, 1)
        all_preds.append(preds)

    mean_preds = np.mean(all_preds, axis=0)
    print(f"  {model_name}: {len(seeds)} seeds 앙상블 완료")
    return mean_preds

seeds = [42, 123, 456, 789, 2024, 2025, 2026, 777, 1234, 5678]

# SVR (시드 불필요하지만 데이터 섞기 효과를 위해 KFold 기반 예측)
print("\n--- SVR Full Training ---")
svr_final = SVR(kernel='rbf', C=10, epsilon=0.01, gamma='scale')
svr_final.fit(X_train_scaled, y_train)
svr_preds = np.clip(svr_final.predict(X_test_scaled), 0, 1)
print(f"  SVR 예측 완료")

svr_poly_final = SVR(kernel='poly', C=5, epsilon=0.01, degree=3, gamma='scale')
svr_poly_final.fit(X_train_scaled, y_train)
svr_poly_preds = np.clip(svr_poly_final.predict(X_test_scaled), 0, 1)
print(f"  SVR(Poly) 예측 완료")

# Ridge
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_train_scaled, y_train)
ridge_preds = np.clip(ridge_final.predict(X_test_scaled), 0, 1)
print(f"  Ridge 예측 완료")

# GBR Seed Ensemble
gbr_preds = seed_ensemble_predict(
    GradientBoostingRegressor,
    dict(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42),
    X_train_scaled, y_train, X_test_scaled, seeds, "GBR"
)

# 추가 모델들
model_preds = {
    'svr_rbf': svr_preds,
    'svr_poly': svr_poly_preds,
    'ridge': ridge_preds,
    'gbr': gbr_preds,
}
model_maes = {
    'svr_rbf': svr_mae,
    'svr_poly': svr_poly_mae,
    'ridge': ridge_mae,
    'gbr': gbr_mae,
}

if HAS_CATBOOST:
    cat_preds = seed_ensemble_predict(
        CatBoostRegressor,
        dict(iterations=1000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
             random_seed=42, verbose=0, loss_function='MAE'),
        X_train_scaled, y_train, X_test_scaled, seeds, "CatBoost"
    )
    model_preds['catboost'] = cat_preds
    model_maes['catboost'] = cat_mae

if HAS_XGB:
    xgb_preds = seed_ensemble_predict(
        XGBRegressor,
        dict(n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
             colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0),
        X_train_scaled, y_train, X_test_scaled, seeds, "XGBoost"
    )
    model_preds['xgboost'] = xgb_preds
    model_maes['xgboost'] = xgb_mae

if HAS_LGBM:
    lgbm_preds = seed_ensemble_predict(
        LGBMRegressor,
        dict(n_estimators=500, learning_rate=0.05, max_depth=5, num_leaves=31,
             subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
             random_state=42, verbose=-1),
        X_train_scaled, y_train, X_test_scaled, seeds, "LightGBM"
    )
    model_preds['lightgbm'] = lgbm_preds
    model_maes['lightgbm'] = lgbm_mae

# ============================================================
# 10. 가중 앙상블 (MAE 역수 기반 가중치)
# ============================================================
print("\n" + "="*60)
print("[가중 앙상블]")
print("="*60)

# MAE가 낮을수록 높은 가중치
weights = {}
total_inv_mae = sum(1.0/mae for mae in model_maes.values())
for name, mae in model_maes.items():
    weights[name] = (1.0/mae) / total_inv_mae
    print(f"  {name}: MAE={mae:.5f}, Weight={weights[name]:.4f}")

# 가중 평균
final_preds = np.zeros(len(X_test))
for name, preds in model_preds.items():
    final_preds += weights[name] * preds

final_preds = np.clip(final_preds, 0, 1)

print(f"\n  최종 예측 분포: mean={final_preds.mean():.4f}, std={final_preds.std():.4f}")
print(f"  Train target 분포: mean={y_train.mean():.4f}, std={y_train.std():.4f}")

# ============================================================
# 11. 제출 파일 생성
# ============================================================
submission['stress_score'] = final_preds
submission.to_csv('/content/submission_ensemble.csv', index=False)
print(f"\n✅ 제출 파일 저장 완료: submission_ensemble.csv")
print(f"   제출 파일 shape: {submission.shape}")
print(submission.head(10))

# ============================================================
# 12. SVR 단독 제출 파일도 생성 (1위 전략 그대로)
# ============================================================
submission_svr = pd.read_csv('/content/sample_submission.csv')
submission_svr['stress_score'] = svr_preds
submission_svr.to_csv('/content/submission_svr_only.csv', index=False)
print(f"\n✅ SVR 단독 제출 파일도 저장: submission_svr_only.csv")

# ============================================================
# 13. 자체 검토 & 요약
# ============================================================
print("\n" + "="*60)
print("[자체 검토 리포트]")
print("="*60)
print("""
📊 데이터 특성:
  - Train 3000행, Test 3000행 (동일 크기)
  - Target: 0~1 연속값 (스트레스 점수)
  - 결측치: medical_history(43%), family_medical_history(50%),
            mean_working(34%), edu_level(20%)
  - 수치형 8개 + 카테고리 7개 원본 피처

🔧 전처리 전략:
  - 결측치: 카테고리→ 'none'/'unknown', 수치형→ 중앙값 + 결측 flag
  - 인코딩: LabelEncoding (SVR 호환)
  - 스케일링: StandardScaler (SVR 필수)

🧪 피처 엔지니어링 (12개 파생변수):
  - BMI, 맥압, 평균동맥압, 혈압비율
  - 고혈압/콜레스테롤/혈당/골밀도 수준 범주화
  - 나이그룹, 비만도, 건강리스크 종합점수
  - 나이*활동량 교호작용, 콜레스테롤*혈당

🤖 모델 전략 (No Free Lunch Theorem):
  - SVR RBF  + SVR Poly
  - Ridge, GBR, CatBoost, XGBoost, LightGBM
  - Seed Ensemble (10 seeds) + MAE 역수 가중 앙상블

⚠️ 보완 포인트:
  1. Optuna로 SVR (C, epsilon, gamma) 추가 튜닝 가능
  2. 스태킹 앙상블 (1단계 모델 → 2단계 메타 모델) 시도 가능
  3. Target Encoding (카테고리→타겟 평균) 추가 가능
  4. 피처 선택 (SHAP/importance 기반) 으로 노이즈 제거 가능
""")

Train: (3000, 18), Test: (3000, 17)
Target 분포: mean=0.482, std=0.288

Combined data: (6000, 16)

[피처 엔지니어링]
  파생변수 생성 완료. 현재 컬럼 수: 31

[결측치 처리]
  결측치 처리 완료. 남은 결측: 0

[카테고리 인코딩]
  인코딩 완료. 최종 컬럼: ['gender', 'age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'bp_ratio', 'hypertension', 'cholesterol_level', 'glucose_level', 'bone_risk', 'age_group', 'obesity', 'health_risk_score', 'activity_num', 'age_activity', 'age_bmi', 'cholesterol_glucose', 'mean_working_missing']

X_train: (3000, 32), X_test: (3000, 32)

[개별 모델 CV 평가]
  SVR (RBF): MAE = 0.20970 (±0.01196)
  SVR (Poly): MAE = 0.25739 (±0.00995)
  Ridge: MAE = 0.24784 (±0.00599)
  GBR: MAE = 0.21314 (±0.00690)
  CatBoost not available, skipping...
  XGBoost: MAE = 0.19901 (±0.00608)
  LightGBM: M

In [11]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import copy, warnings
warnings.filterwarnings('ignore')


print("="*60, flush=True)
print(" v3: 검증된 개선만 적용", flush=True)
print("="*60, flush=True)

train = pd.read_csv('/mnt/user-data/uploads/train.csv')
test = pd.read_csv('/mnt/user-data/uploads/test.csv')
sub = pd.read_csv('/mnt/user-data/uploads/sample_submission.csv')
target = train['stress_score'].copy()
y = target.values
n_train = len(train)
train.drop(['ID','stress_score'], axis=1, inplace=True)
test.drop(['ID'], axis=1, inplace=True)
data = pd.concat([train, test], ignore_index=True)

# --- 전처리: v1에서 잘 됐던 것 유지 + 결측 플래그만 추가 ---
data['bmi'] = data['weight'] / ((data['height']/100)**2)
data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
data['map_bp'] = data['diastolic_blood_pressure'] + data['pulse_pressure']/3
data['bp_ratio'] = data['systolic_blood_pressure'] / (data['diastolic_blood_pressure']+1e-5)
data['hypertension'] = ((data['systolic_blood_pressure']>=140)|(data['diastolic_blood_pressure']>=90)).astype(int)
data['age_group'] = pd.cut(data['age'], bins=[0,30,45,60,75,100], labels=[0,1,2,3,4]).astype(int)
act_map = {'light':0,'moderate':1,'intense':2}
data['activity_num'] = data['activity'].map(act_map)
data['age_activity'] = data['age'] * data['activity_num']
data['age_bmi'] = data['age'] * data['bmi']

# 결측 플래그 (v2에서 검증)
data['edu_missing'] = data['edu_level'].isna().astype(int)
data['mw_missing'] = data['mean_working'].isna().astype(int)

data['medical_history'] = data['medical_history'].fillna('none')
data['family_medical_history'] = data['family_medical_history'].fillna('none')
data['edu_level'] = data['edu_level'].fillna('unknown')
data['mean_working'] = data['mean_working'].fillna(data['mean_working'].median())

cat_cols = ['gender','activity','smoke_status','medical_history',
            'family_medical_history','sleep_pattern','edu_level']
for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

print(f"피처: {data.shape[1]}개", flush=True)

X_train = data.iloc[:n_train].values
X_test = data.iloc[n_train:].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X_train)
Xt = scaler.transform(X_test)

# --- 모델: 정규화 강화 버전 ---
models = {
    'svr': SVR(kernel='rbf', C=10, epsilon=0.01, gamma='scale'),
    'xgb': XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5,
        min_child_weight=5, random_state=42, verbosity=0),
    'cat': CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=5,
        l2_leaf_reg=5, random_seed=42, verbose=0, loss_function='MAE'),
    'lgbm': LGBMRegressor(n_estimators=800, learning_rate=0.03, max_depth=4,
        num_leaves=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.5, min_child_samples=15,
        random_state=42, verbose=-1),
    'gbr': GradientBoostingRegressor(n_estimators=500, learning_rate=0.03,
        max_depth=3, subsample=0.8, min_samples_leaf=10, random_state=42),
}

# --- Stacking L1: 10-Fold ---
print("\n[L1 OOF]", flush=True)
kf = KFold(n_splits=10, shuffle=True, random_state=42)
oof_tr = np.zeros((n_train, len(models)))
oof_te = np.zeros((len(X_test), len(models)))

for i, (name, model) in enumerate(models.items()):
    folds_te = []
    for fold, (ti, vi) in enumerate(kf.split(Xs)):
        m = copy.deepcopy(model)
        m.fit(Xs[ti], y[ti])
        oof_tr[vi, i] = np.clip(m.predict(Xs[vi]), 0, 1)
        folds_te.append(np.clip(m.predict(Xt), 0, 1))
    oof_te[:, i] = np.mean(folds_te, axis=0)
    print(f"  {name:8s} MAE: {mean_absolute_error(y, oof_tr[:,i]):.5f}", flush=True)

# --- Stacking L2: 3개 메타 모델 블렌딩 ---
print("\n[L2 Meta]", flush=True)

# Ridge
meta1 = RidgeCV(alphas=np.logspace(-3,2,50))
meta1.fit(oof_tr, y)
p1_oof = np.clip(meta1.predict(oof_tr), 0, 1)
p1_te = np.clip(meta1.predict(oof_te), 0, 1)
print(f"  Ridge MAE: {mean_absolute_error(y, p1_oof):.5f}  coefs={dict(zip(models.keys(), meta1.coef_.round(3)))}", flush=True)

# SVR meta
ms = StandardScaler()
oof_tr_s = ms.fit_transform(oof_tr)
oof_te_s = ms.transform(oof_te)
meta2 = SVR(kernel='rbf', C=5, epsilon=0.005)
meta2.fit(oof_tr_s, y)
p2_oof = np.clip(meta2.predict(oof_tr_s), 0, 1)
p2_te = np.clip(meta2.predict(oof_te_s), 0, 1)
print(f"  SVR   MAE: {mean_absolute_error(y, p2_oof):.5f}", flush=True)

# XGB meta
meta3 = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=2,
    subsample=0.8, reg_lambda=3.0, random_state=42, verbosity=0)
meta3.fit(oof_tr, y)
p3_oof = np.clip(meta3.predict(oof_tr), 0, 1)
p3_te = np.clip(meta3.predict(oof_te), 0, 1)
print(f"  XGB   MAE: {mean_absolute_error(y, p3_oof):.5f}", flush=True)

# OOF MAE로 최적 블렌딩 비율 탐색
best_mae = 999
best_w = None
for w1 in np.arange(0, 1.05, 0.05):
    for w2 in np.arange(0, 1.05-w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < -0.01: continue
        blend_oof = w1*p1_oof + w2*p2_oof + w3*p3_oof
        mae = mean_absolute_error(y, np.clip(blend_oof, 0, 1))
        if mae < best_mae:
            best_mae = mae
            best_w = (round(w1,2), round(w2,2), round(w3,2))

print(f"  Best blend: w={best_w}, MAE={best_mae:.5f}", flush=True)

meta_pred = best_w[0]*p1_te + best_w[1]*p2_te + best_w[2]*p3_te
meta_pred = np.clip(meta_pred, 0, 1)

# --- Seed Ensemble (XGB + CatBoost) ---
print("\n[Seed Ensemble]", flush=True)
seeds = [42,123,456,789,2024,2025,2026,777,1234,5678]

def seed_pred(Cls, params, Xs, y, Xt, seeds):
    ap = []
    for s in seeds:
        p = params.copy()
        if 'random_seed' in p: p['random_seed'] = s
        else: p['random_state'] = s
        m = Cls(**p); m.fit(Xs, y)
        ap.append(np.clip(m.predict(Xt), 0, 1))
    return np.mean(ap, axis=0)

xgb_s = seed_pred(XGBRegressor,
    dict(n_estimators=800, learning_rate=0.03, max_depth=4,
         subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5,
         min_child_weight=5, random_state=42, verbosity=0),
    Xs, y, Xt, seeds)
print("  XGB done", flush=True)

cat_s = seed_pred(CatBoostRegressor,
    dict(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=5,
         random_seed=42, verbose=0, loss_function='MAE'),
    Xs, y, Xt, seeds)
print("  Cat done", flush=True)

# --- 최종 블렌딩: OOF로 비율 최적화 불가하므로 보수적 ---
final = 0.50*meta_pred + 0.25*xgb_s + 0.25*cat_s
final = np.clip(final, 0, 1)

print(f"\n  Final: mean={final.mean():.4f} std={final.std():.4f}", flush=True)

sub['stress_score'] = final
sub.to_csv('/mnt/user-data/outputs/submission_v3_final.csv', index=False)
print("✅ submission_v3_final.csv", flush=True)

# 메타만 버전도 저장
s2 = pd.read_csv('/mnt/user-data/uploads/sample_submission.csv')
s2['stress_score'] = meta_pred
s2.to_csv('/mnt/user-data/outputs/submission_v3_meta.csv', index=False)
print("✅ submission_v3_meta.csv", flush=True)

ModuleNotFoundError: No module named 'catboost'